In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# 1. 读取数据
# =========================
file_path = "BAT_Heat_Log_Data_2026_05_21_16_35_17.csv"   # ← 改成你的文件名
df = pd.read_csv(file_path)

# 清理列名
df.columns = df.columns.str.strip()
df = df.replace(r'^\s*$', np.nan, regex=True)

# =========================
# 2. 时间解析
# =========================
df['UTC_Time'] = pd.to_datetime(df['UTC_Time'], format='%Y_%m_%d_%H_%M_%S.%f')

# 按时间排序（防止乱序）
df = df.sort_values('UTC_Time')

# =========================
# 3. 采样间隔分析
# =========================
df['dt'] = df['UTC_Time'].diff().dt.total_seconds()

print("=== 采样间隔统计 ===")
print(df['dt'].describe())

# 可视化
plt.figure(figsize=(10,4))
plt.plot(df['dt'])
plt.title("Sampling Interval (Original)")
plt.ylabel("Seconds")
plt.grid()
plt.show()

# =========================
# 4. 判断是否需要插值
# =========================
dt_std = df['dt'].std()
print("采样间隔标准差:", dt_std)

need_resample = dt_std > 0.002   # 阈值（2ms）

print("是否需要重采样:", need_resample)

# =========================
# 5. 执行重采样（统一 50Hz）
# =========================
df = df.set_index('UTC_Time')

df_resampled = df.resample('20ms').mean()

# 插值
df_resampled = df_resampled.interpolate(method='linear')

# =========================
# 6. 构造“正常横坐标”（均匀时间轴）
# =========================
df_resampled['time_sec'] = np.arange(len(df_resampled)) * 0.02

# =========================
# 7. 只保留 IMU 数据
# =========================
imu = df_resampled[[
    'time_sec',
    'Accel_x','Accel_y','Accel_z',
    'gyro_x','gyro_y','gyro_z'
]].copy()

# =========================
# 8. 单位转换（Accel: mg → m/s²）
# =========================
imu[['Accel_x','Accel_y','Accel_z']] = imu[['Accel_x','Accel_y','Accel_z']] / 1000.0 * 9.81

# =========================
# 9. 检查重采样效果
# =========================
check_dt = imu['time_sec'].diff()

plt.figure(figsize=(10,4))
plt.plot(check_dt)
plt.title("Sampling Interval After Resample")
plt.grid()
plt.show()

# =========================
# 10. 输出新的 CSV
# =========================
output_path = "imu_processed.csv"
imu.to_csv(output_path, index=False)

print("✅ 已生成文件:", output_path)